<a href="https://colab.research.google.com/github/p-tech/ntu-growth-hacking/blob/main/sentiment/Notebook_2_Comparing_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 · Do three sentiment methods tell the same story?
**NTU MSc Growth Hacking · Customer Insight Lab · 45–55 minutes**

The Oddbox founder wants to know how people talk about veg boxes online. Someone suggests reporting the percentage of positive messages. Would you give the founder the same figure whichever tool you used?

In Notebook 1 you compared your reading with a neighbour’s. Here you will read three messages yourself, then compare **TextBlob, VADER and RoBERTa on exactly the same messages**. You will finish with a figure, a quotation and a growth experiment.

Save a copy in Colab. Run one cell at a time using its play button. Stop at each discussion prompt. Choose **Upload CSV** for the class file supplied by your tutor. The alternative practice messages are fictional. You can complete the whole activity with them. For real messages, use an English-language Apify CSV supplied by your tutor or collected by your group. The model download needs an internet connection.

## 1 · Which messages would answer the founder’s question?
For your own collection, choose one platform and one type of message, such as comments under public posts about veg boxes. Search for **Oddbox**, **veg box delivery**, **wonky veg box** or **#vegbox**. Inspect results for relevance before choosing your search. A broad hashtag can lead to growers and recipes as well as customers. Searching for “veg box complaint” is likely to change the balance again.

In Apify, choose a scraper for your chosen platform that accepts your search term or the relevant post URLs. Collect around 30–60 short English messages from a defined period. Check the preview. Each row should contain one comment or post. Export the dataset as CSV with the message text and, if available, its unique comment or post ID. For this activity, exclude adverts and brand-authored posts when preparing your file.

**Record your choices:** Platform or practice set ___ · search phrase ___ · dates ___ · whose views might be missing? ___

In [ ]:
%pip install emoji textblob vaderSentiment "transformers>=5,<6" "huggingface-hub>=1.32,<3"

In [ ]:
import io
import re
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', None)
source = "Upload CSV" #@param ["Upload CSV", "Practice messages"]
if source == "Upload CSV":
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError('Choose one CSV file, then run this cell again.')
    raw = pd.read_csv(io.BytesIO(next(iter(uploads.values()))), dtype=str, keep_default_na=False)
else:
    messages = ["Great, another soggy box on the doorstep 🙄","The pears looked odd but tasted brilliant.","Does anyone know when next week's list comes out?","Delivery was late. Again.","Love the idea. Can't get through another cabbage though.","I wish I'd tried the small box sooner!","Is the fruit booster available every week?","The recipe used every last carrot. Keeping that one.","Not happy about the change of delivery day.","That muddy potato? Best roastie I've made in ages.","My box contained leeks, squash and three onions.","No more guessing what to cook on Thursday 🙂","I cancelled after the third missed delivery.","Fresh veg, shame about the split bag.","NOT impressed!!!","I thought it would be pricey. Actually sorted four dinners for us.","We get ours every other week.","Another bag of kale. Lucky me.","Kids asked for seconds. That's a first!","Could we swap beetroot for something else?","Arrived at 6 am as promised.","The delivery photo showed somebody else's door.","The box is good value when we use everything.","Lovely selection but too much for one person.","No complaints here!","Why send a recipe that needs ingredients outside the box?","The driver moved it out of the rain. Thank you!","I can pause a week in the account settings.","Beautiful apples. Would happily buy those again.","Half the spinach was already yellow.","Is that £8 just for the first box?","£16 felt fair once I worked out the cost per meal.","My neighbour gets the same box on Thursdays.","Ordering was easy. Cancelling took three emails.","The small size works perfectly for our flat.","Three courgettes and no idea what to do with them 😬"]
    raw = pd.DataFrame({'comment_id': range(1, len(messages)+1), 'text': messages})
    raw = pd.concat([raw, raw.iloc[[0]], pd.DataFrame([{'comment_id': 37, 'text': '  '}])], ignore_index=True)
print('Columns:', list(raw.columns))
display(raw.head(4))

## 2 · What should count as one message?
Read the headings printed above. Enter the name of the column containing the customer's actual words. For an upload, choose the unique ID for the **message itself** if available. A post ID shared by several comments would be unsuitable. Leave the ID field empty if unsure.

A blank message gives us no words to judge. A repeated import gives one voice extra weight. The cell trims spare spaces and removes repeats only when both ID and wording match. With no ID it keeps repeated wording for inspection, because two people might write the same words. Look at the counts before continuing. For a large upload we use a repeatable random sample of up to 60 usable messages to keep the activity manageable.

In [ ]:
text_column = "caption" #@param {type:"string"}
id_column = "" #@param {type:"string"}
if source == 'Practice messages':
    text_column, id_column = 'text', 'comment_id'
if text_column not in raw or (id_column and id_column not in raw):
    raise ValueError('Copy the exact column name from the preview above.')
data = pd.DataFrame({'original': raw[text_column].fillna('').astype(str)})
data['message'] = data.original.str.split().str.join(' ')
data['source_id'] = raw[id_column].fillna('').astype(str).str.strip() if id_column else ''
blank = data.message.eq('')
repeat = ~blank & data.source_id.ne('') & data.duplicated(['source_id', 'message'])
clean = data.loc[~(blank | repeat)].copy()
print(f'{len(raw)} imported, {blank.sum()} blank, {repeat.sum()} repeated imports, {len(clean)} usable.')
display(clean.loc[clean.message.duplicated(keep=False), ['source_id','message']].head(6))
if clean.empty:
    raise ValueError('No usable messages. Check the text column or upload another file.')
if len(clean) > 60:
    clean = clean.sample(60, random_state=42).sort_index()
results = clean.reset_index(drop=True)
results.insert(0, 'row', range(1, len(results)+1))
display(results[['row','message']].head(3))

**Your turn.** Before running a method, give each of the first three messages one overall label in your notes: positive, negative or mixed (neutral). Which words or punctuation shaped your most difficult decision? Later, compare your choice with all three outputs. The automated methods call the middle category *neutral*. That label alone cannot tell us whether a message expresses balanced feelings or simply asks a question.

**Why keep “not”, capitals and emojis?** Compare “impressed” with “NOT impressed!!!”. Removing common words and punctuation can remove the clues a reader uses. Stemming shortens related words to a common stem. Lemmatisation uses grammar and vocabulary to find their dictionary form. For example, “delivered” and “delivering” can both become “deliver”. This can help group messages by subject in a topic model. Here we keep the sentence intact so each method can use the clues it recognises.

## 3 · TextBlob and VADER: what changes when rules change?
**TextBlob** looks up words and phrases associated with opinions and produces an overall polarity score from −1 to +1. In this exercise a score above zero becomes positive, below zero becomes negative, and zero becomes neutral. Even a very small non-zero score therefore earns a direction.

**VADER** also uses a word list, with rules designed for short social messages. Capitals, punctuation, intensifiers and negation can affect its compound score. Its usual rule gives positive at +0.05 or above and negative at −0.05 or below. The gap in between is neutral. A score of +0.04 would therefore receive different labels under our two decision rules even if both methods produced that number.

Run the cell. Find a message where the two methods agree on direction. Do the score magnitudes tell you which judgement is more trustworthy? Each method constructs its score differently.

**What do the extra scores mean?** TextBlob's subjectivity runs from 0 to 1. A higher value means the wording appears more opinion-based. It cannot tell us whether a claim is true or how confident the model is. “Awful service” and “wonderful service” both express strong opinions, but their polarity points in opposite directions.

VADER's **pos**, **neu** and **neg** show the relative shares of the text it scores as positive, neutral and negative. They add to roughly 1. A message can have both positive and negative shares. The **compound** score combines sentiment with rules about emphasis and context. Use compound to assign the overall label. Look for a message with both praise and criticism. What does the breakdown reveal that its single label hides?

**Read two scores together.** Read polarity −0.20 and subjectivity 0.80 together as a mildly negative opinion. 0.80 does not mean the model is 80% sure. A low value can mean it missed an implied opinion.

**Try the boundary.** −0.48 gives negative. +0.04 gives neutral. +0.05 gives positive. Negative starts at −0.05 or below. −0.48 is a score, not a percentage of customers. If the compound score moves from +0.04 to +0.05, how much changed in the score? What changed in the reported label?

In [ ]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()
def word_scores(text):
    blob = TextBlob(text).sentiment
    v = vader.polarity_scores(text)
    return {'blob_polarity': blob.polarity, 'blob_subjectivity': blob.subjectivity,
            'TextBlob': 'positive' if blob.polarity > 0 else 'negative' if blob.polarity < 0 else 'neutral',
            **{'vader_' + key: value for key, value in v.items()},
            'VADER': 'positive' if v['compound'] >= .05 else 'negative' if v['compound'] <= -.05 else 'neutral'}

word_table = pd.DataFrame(results.message.map(word_scores).tolist(), index=results.index)
results[word_table.columns] = word_table
display(results[['row','message'] + list(word_table.columns)].head(6))

## 4 · What does RoBERTa pick up from context?
This version of RoBERTa learned patterns in English-language tweets and was trained to classify sentiment. It reads words in relation to their neighbours. For each message it assigns a score to each of three labels and picks the largest. If two scores are close, inspect the wording before interpreting the winner. A high score can still accompany a poor reading of sarcasm.

The first run downloads the Cardiff NLP model and may take a few minutes. The cell prepares links and account mentions in the style the model expects. Very long messages are cut to the model input limit of 512 tokens (pieces of words and punctuation). Use short messages for this comparison.

**Illustrative scores.** Negative 0.46, neutral 0.44, positive 0.10 gives negative. Its lead over neutral is just 0.02. Read the message before reporting a firm judgement. 0.89 negative versus 0.08 neutral and 0.03 positive gives a margin of 0.81. The model favours negative more strongly here. Even a large margin can accompany a mistaken reading. These scores describe the model’s preference for this wording. They do not measure how strongly the customer feels.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device).eval()
labels = ['negative', 'neutral', 'positive']
model_labels = [model.config.id2label[i].lower() for i in range(3)]
if set(model_labels) != set(labels):
    raise ValueError('Unexpected model labels. Check the model name.')
def roberta_scores(messages):
    texts = [re.sub(r'https?://\S+', 'http', re.sub(r'(?<!\w)@\w+', '@user', t)) for t in messages]
    scores = []
    with torch.inference_mode():
        for start in range(0, len(texts), 8):
            inputs = tokenizer(texts[start:start+8], padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
            scores.extend(model(**inputs).logits.softmax(dim=1).cpu().tolist())
    return pd.DataFrame(scores, columns=model_labels, index=messages.index)

probabilities = roberta_scores(results.message)
results['RoBERTa'] = probabilities.idxmax(axis=1)
for label in labels:
    results['roberta_' + label] = probabilities[label]
methods = ['TextBlob', 'VADER', 'RoBERTa']
results['roberta_margin'] = probabilities.apply(lambda row: row.nlargest(2).diff().abs().iloc[-1], axis=1)
display(results[['row','message'] + methods + ['roberta_' + label for label in labels] + ['roberta_margin']].head(3).round(3))

## 5 · Would the founder hear a different story?
Compare the three rows in the table below. Each row describes the same sample. Read **across** to see how one method divides the messages. Read **down** the positive percentage column to see how sensitive the headline is to the method.

For example, if two methods labelled 12 and 18 out of 30 messages positive, their positive shares would be 40% and 60%. That 20 percentage point gap would come from the analysis method. The customers’ words stayed the same.

In [ ]:
counts = pd.DataFrame({m: results[m].value_counts().reindex(labels, fill_value=0) for m in methods}).T
percentages = counts.div(len(results)).mul(100)
display(counts.add_suffix(' count').join(percentages.round(1).add_suffix(' %')))
percentages.plot.bar(rot=0, ylabel='Share of analysed messages (%)', ylim=(0,100), title=f'Sentiment across {len(results)} messages')
results['all_agree'] = results[methods].nunique(axis=1).eq(1)
print(f'All three agree on {results.all_agree.sum()} of {len(results)} messages.')

def disagreement(labels_row):
    counts = labels_row.value_counts()
    if len(counts) == 1:
        return 'All agree'
    if len(counts) == 3:
        return 'All three differ'
    return labels_row.index[labels_row != counts.index[0]][0] + ' differs'

results['model_comparison'] = results[methods].apply(disagreement, axis=1)
different = results.loc[~results.all_agree].sort_values(['model_comparison','row'])
if different.empty:
    print('All three agree throughout. Choose a label you would question.')
display((different if len(different) else results)[['row','message','model_comparison'] + methods])


**Work with a neighbour.** First compare your own three judgements from Block 2 with the model labels in Block 4. Then choose one row from Block 5. Read it aloud. Which method gives the most convincing reading **for this message**, and which words support your view? Two methods agreeing is a reason to look more closely, not proof of accuracy.

**Your briefing to the founder**

1. “Of our ___ messages, the positive share ranged from ___% with ___ to ___% with ___.”
2. “In row ___, the phrase ‘___’ suggests ___, although the methods labelled it ___.”
3. “This suggests we could test ___ with ___ customers and measure ___.” Explain how your search phrase or practice data limits the claim.

For example, questions about seeing box contents could justify testing a contents preview. Compare the proportion of visitors adding a box to their basket. A neutral question can still reveal something worth testing.

## 6 · Does cleaning change what the founder hears?
Block 2 removed blank rows and repeat imports. Here you change the **wording** on a copy. Keep the original scores as your starting point. Choose one setting below. “Keep wording” is a control: you should get the same scores twice.

Before running the cell, pick one message and predict what will happen. Would removing the exclamation marks from “NOT impressed!!!” weaken the negative score enough to change the label? Would changing capitals have the same effect? Try one setting at a time so you can explain any difference.

In [ ]:
import unicodedata
cleaning = "Remove emojis" #@param ["Keep wording", "Use lower case", "Remove punctuation", "Remove emojis"]
def change_text(text):
    if cleaning == 'Use lower case':
        return text.lower()
    if cleaning == 'Remove punctuation':
        # Keep apostrophes inside contractions such as isn't.
        text = ''.join(' ' if unicodedata.category(c).startswith('P') and c not in "'’" else c for c in text)
    if cleaning == 'Remove emojis':
        import emoji
        text = emoji.replace_emoji(text, replace=' ')
    return ' '.join(text.split())

comparison = results.copy()
comparison['cleaning'] = cleaning
comparison['changed_text'] = results.message.map(change_text)
valid = comparison.changed_text.ne('')
altered = valid & comparison.changed_text.ne(results.message)
score_cols = list(word_table.columns) + ['RoBERTa'] + ['roberta_' + label for label in labels]
after = results[score_cols].copy()
if altered.any():
    messages = comparison.loc[altered, 'changed_text']
    new_words = pd.DataFrame(messages.map(word_scores).tolist(), index=messages.index)
    after.loc[altered, new_words.columns] = new_words
    new_probs = roberta_scores(messages)
    after.loc[altered, 'RoBERTa'] = new_probs.idxmax(axis=1)
    for label in labels:
        after.loc[altered, 'roberta_' + label] = new_probs[label]
after.loc[~valid, :] = None
comparison = comparison.join(after.add_suffix('_after'))
print(f'{altered.sum()} messages changed and still contain text. {sum(~valid)} became blank and are excluded from both sides of the comparison.')
summary = []
for method in methods:
    before_labels = results.loc[valid, method]
    after_labels = after.loc[valid, method]
    summary.append({'method': method, 'paired messages': valid.sum(),
                    'labels changed': (before_labels != after_labels).sum(),
                    'positive before (%)': 100 * before_labels.eq('positive').mean(),
                    'positive after (%)': 100 * after_labels.eq('positive').mean()})
display(pd.DataFrame(summary).round(1))
inspect = comparison.loc[altered] if altered.any() else comparison
for method, metrics in [('TextBlob',['blob_polarity','blob_subjectivity']),
                        ('VADER',['vader_neg','vader_neu','vader_pos','vader_compound']),
                        ('RoBERTa',['roberta_negative','roberta_neutral','roberta_positive'])]:
    pairs = [col for key in [method] + metrics for col in [key, key + '_after']]
    display(inspect[['row','message','changed_text'] + pairs].head(5).round(3))

**Make a decision.** Pick one row and describe the edit, the score before and after, and whether the label changed. Read both versions aloud. Which version better preserves the customer's meaning? If scores stayed the same, check whether the selected edit actually changed that message. Tell your neighbour which cleaning choice you would use on this dataset and why. The export below records the most recent choice.

## 7 · Download and investigate a disagreement
Choose Excel to download one workbook. Open **Disagreements** first. Each row shows the message, which method differs and the three labels side by side. Use the column filters to find, for example, every row where VADER differs. **All messages** contains the full dataset. **Score guide** explains each number. The CSV option gives you the full dataset with the same comparison columns.

The disagreement sheet includes rows where models differ on either wording. The two comparison columns tell you whether the difference occurs on the original message, after cleaning, or both. Pick one row. Quote the words that support your reading and use one score to explain why the model’s label deserves a closer look.

In [ ]:
from google.colab import files
download_format = "Excel workbook" #@param ["Excel workbook", "Full CSV"]
export = comparison.copy()
after_cols = [m + '_after' for m in methods]
valid_after = export[after_cols].notna().all(axis=1)
export['comparison_after'] = 'Empty after cleaning'
if valid_after.any():
    revised = export.loc[valid_after, after_cols].set_axis(methods, axis=1)
    export.loc[valid_after, 'comparison_after'] = revised.apply(disagreement, axis=1)
first = ['row','message','model_comparison'] + methods + ['changed_text','comparison_after'] + after_cols
export = export[first + [col for col in export if col not in first]]
disputes = export.loc[(export.model_comparison != 'All agree') |
                     (valid_after & export.comparison_after.ne('All agree'))]
if download_format == 'Full CSV':
    filename = 'notebook-2-sentiment-comparison.csv'
    export.to_csv(filename, index=False, encoding='utf-8-sig')
else:
    from openpyxl.styles import Alignment
    filename = 'notebook-2-sentiment-comparison.xlsx'
    guide = pd.DataFrame([['TextBlob polarity', '−1 to +1. The sign sets the label in this notebook.', 'A polarity of −0.20 means slightly negative wording.'], ['TextBlob subjectivity', '0 to 1. Higher means TextBlob picks up more opinion-based wording.', 'Read polarity −0.20 and subjectivity 0.80 together as a mildly negative opinion. 0.80 does not mean the model is 80% sure. A low value can mean it missed an implied opinion.'], ['VADER neg, neu, pos', 'Shares of the wording VADER associates with each category. They sum to about 1.', 'Look for both positive and negative shares in a message such as “Lovely fruit, shame about the delivery”. The shares help you spot competing signals.'], ['VADER compound', '−1 to +1. This combines word scores and rules for emphasis and negation.', '−0.48 gives negative. +0.04 gives neutral. +0.05 gives positive. Negative starts at −0.05 or below. −0.48 is a score, not a percentage of customers.'], ['RoBERTa label scores', 'Three values between 0 and 1 that sum to approximately 1. The largest selects the label.', 'Negative 0.46, neutral 0.44, positive 0.10 gives negative. Its lead over neutral is just 0.02. Read the message before reporting a firm judgement.'], ['RoBERTa winning margin', 'Highest score minus second-highest score. A smaller gap means the two leading labels were closer.', '0.89 negative versus 0.08 neutral and 0.03 positive gives a margin of 0.81. The model favours negative more strongly here. Even a large margin can accompany a mistaken reading.']], columns=['Score','How to read it','Illustrative example'])
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        for title, table in [('Disagreements', disputes), ('All messages', export), ('Score guide', guide)]:
            table.to_excel(writer, sheet_name=title, index=False)
            sheet = writer.sheets[title]
            sheet.freeze_panes = 'C2' if title != 'Score guide' else 'A2'
            sheet.auto_filter.ref = sheet.dimensions
            for column in sheet.columns:
                heading = column[0].value
                width = 64 if heading in ['message','original','changed_text','How to read it','Illustrative example'] else max(18,len(heading)+3)
                sheet.column_dimensions[column[0].column_letter].width = width
                if width == 64:
                    for cell in column[1:]:
                        cell.alignment = Alignment(wrap_text=True, vertical='top')
print(f'{len(disputes)} of {len(export)} messages show a model disagreement before or after cleaning.')
files.download(filename)

### Method references

[TextBlob sentiment guide](https://textblob.readthedocs.io/en/dev/quickstart.html#sentiment-analysis) · [VADER scoring rules](https://github.com/cjhutto/vaderSentiment#about-the-scoring) · [Cardiff NLP model card](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)